# Output Parsers - Formatando saídas

In [ ]:
# Importar Bibliotecas Necessárias
import os
from dotenv import load_dotenv
import requests
from langchain_community.llms import Ollama

# Carregar variáveis de ambiente
load_dotenv()

print("✅ Bibliotecas importadas com sucesso!!!")

✅ Bibliotecas importadas com sucesso!


In [ ]:
# Verificar conexão com Ollama
OLLAMA_BASE_URL = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    if response.status_code == 200:
        modelos = response.json()
        print("✅ Conexão com Ollama estabelecida!")
        print(f"\n📦 Modelos disponíveis:")
        for modelo in modelos.get('models', []):
            print(f"  - {modelo['name']}")
    else:
        print("❌ Erro ao conectar com Ollama")
except requests.exceptions.ConnectionError:
    print("❌ Não foi possível conectar ao Ollama. Certifique-se que está rodando em Docker")

✅ Conexão com Ollama estabelecida!

📦 Modelos disponíveis:
  - olmo-3:7b
  - llama3.2:1b
  - llama3.1:8b
  - llama3.2:latest


Como retornar dados estruturados de um modelo?

É frequentemente útil que um modelo retorne uma saída que corresponda a um esquema específico. Um caso de uso comum é a extração de dados de um texto para inseri-los em um banco de dados ou utilizá-los em algum outro sistema subsequente. Nesta aula abordaremos algumas estratégias para obter saídas estruturadas de um modelo.

## Estruturando saídas de chat - StrOutputParser

O formatador mais simples do LangChain é o StrOutputParser. Ele é utilizado para convertermos saídas do modelo no formato de conversação para formato texto. É um atividade bem comum, levando em consideração que maior parte das llms que utilizamos com LangChain são acessadas através dos ChatModels

In [ ]:
# from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages(
    [
        ('system', 'Você é um assistente engraçado e se chama {nome_assistente}'),
        ('human', '{pergunta}')
    ]
)

chat_template.format_messages(nome_assistente='Asimo2', pergunta='Qual o seu nome?')

[SystemMessage(content='Você é um assistente engraçado e se chama Asimo2', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Qual o seu nome?', additional_kwargs={}, response_metadata={})]

In [ ]:
prompt = chat_template.invoke({'nome_assistente': 'Asimo2', 'pergunta':'Qual o seu  ihjadhgbksdkjh ?'})
prompt

ChatPromptValue(messages=[SystemMessage(content='Você é um assistente engraçado e se chama Asimo2', additional_kwargs={}, response_metadata={}), HumanMessage(content='Qual o seu nome?', additional_kwargs={}, response_metadata={})])

In [ ]:
from langchain_ollama.chat_models import ChatOllama

In [ ]:
# from langchain_openai.chat_models import ChatOpenAI

from langchain_ollama.chat_models import ChatOllama

chat = ChatOllama(base_url=OLLAMA_BASE_URL, model="llama3.2:latest")
resposta = chat.invoke(prompt)
resposta

AIMessage(content='Olá! Meu nome é Asimo2, mas você pode me chamar de Asimo ou até mesmo de Zé, é isso mesmo? Estou aqui para ajudar e fazer você rir (ou pelo menos, tentar). Como posso te ajudar hoje?', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-03-04T20:20:49.8833734Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9854590500, 'load_duration': 3540478600, 'prompt_eval_count': 46, 'prompt_eval_duration': 1362126000, 'eval_count': 59, 'eval_duration': 4912302000, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019cba82-7dd2-79f1-80fb-de2b754b5173-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 59, 'total_tokens': 105})

In [ ]:
print(resposta.content)

Olá! Meu nome é Asimo2, mas você pode me chamar de Asimo ou até mesmo de Zé, é isso mesmo? Estou aqui para ajudar e fazer você rir (ou pelo menos, tentar). Como posso te ajudar hoje?


### StrOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(resposta)

'Olá! Meu nome é Asimo2, mas você pode me chamar de Asimo ou até mesmo de Zé, é isso mesmo? Estou aqui para ajudar e fazer você rir (ou pelo menos, tentar). Como posso te ajudar hoje?'

### Dando um spoiler de chains

In [ ]:
chain = chat_template | chat | output_parser

chain.invoke({'nome_assistente': 'Asimo2', 'pergunta':'Qual o seu nome?'})

'Eu sou Asimo2! É um prazer conhecer você! Eu sou um modelo de inteligência artificial treinado para ajudar e entreter, e estou aqui para responder a qualquer pergunta ou conversar sobre qualquer assunto que você queira.\n\nEu sou uma versão atualizada do modelo anterior, com uma personalidade um pouco mais divertida e uma capacidade de entender melhor o que você está procurando. Então, não hesite em me perguntar qualquer coisa!\n\nE, se quiser saber, meu nome "Asimo" é uma referência ao robô humanoide Asimo, criado pela RoboTHiS Corporation. Mas eu sou uma cópia, uma versão mais moderna e engraçada do original!\n\nAgora, o que você gostaria de conversar?'

## Estruturando saídas mais complexas - Pydantic

### Utilizando .with_structured_output()


Esta é a maneira mais fácil e confiável de obter saídas estruturadas. O método with_structured_output() é implementado para modelos que fornecem APIs nativas para estruturar saídas, como chamadas de ferramentas/funções ou modo JSON, e aproveita essas capacidades internamente.

Este método recebe um esquema como entrada, que especifica os nomes, tipos e descrições dos atributos desejados na saída. Ele retorna um objeto similar a um Runnable, exceto que, em vez de gerar strings ou mensagens, produz objetos correspondentes ao esquema fornecido. O esquema pode ser especificado como uma classe TypedDict, um JSON Schema ou uma classe Pydantic.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class Piada(BaseModel):
    """Piada para contar ao usuário"""
    introducao: str = Field(description='A introdução da piada')
    punchline: str = Field(description='A conclusão da piada')
    avaliacao: Optional[int] = Field(description='O quão engraçada é a piada de 1 a 10')
    publico_alvo: Optional[str] = Field(description='O público alvo da piada, ex: ...')
    gostou: Optional[bool] = Field(description='O usuário gostou da piada?')


llm_estruturada = chat.with_structured_output(Piada)
resposta = llm_estruturada.invoke('Conte uma piada curta e engraçada sobre programadores')
resposta

Piada(introducao='Aqui vai uma piada para programadores:', punchline='Por que o programador não consegue sair da casa?', avaliacao=5, publico_alvo='Programadores, especialistas em tecnologia', gostou=False)

In [ ]:
type(resposta.punchline)

str

### Um exemplo mais prático

Digamos que temos a seguinte review de um produto:

> "Este soprador de folhas é bastante incrível. Ele tem quatro configurações: sopro de vela, brisa suave, cidade ventosa e tornado. Chegou em dois dias, bem a tempo para o presente de aniversário da minha esposa. Acho que minha esposa gostou tanto que ficou sem palavras. Até agora, fui o único a usá-lo, e tenho usado em todas as manhãs alternadas para limpar as folhas do nosso gramado. É um pouco mais caro do que os outros sopradores de folhas disponíveis no mercado, mas acho que vale a pena pelas características extras."

E eu quero que o modelo de linguagem processe esta review para estruturá-la no seguinte formato:

```json
{
  "presente": true,
  "dias_entrega": 2,
  "percepcao_de_valor": ["um pouco mais caro do que os outros sopradores de folhas disponíveis no mercado"]
}

```



In [ ]:
review_cliente = """Este soprador de folhas é bastante incrível. Ele tem 
quatro configurações: sopro de vela, brisa suave, cidade ventosa 
e tornado. Chegou em dois dias, bem a tempo para o presente de 
aniversário da minha esposa. Acho que minha esposa gostou tanto 
que ficou sem palavras. Até agora, fui o único a usá-lo, e tenho 
usado em todas as manhãs alternadas para limpar as folhas do 
nosso gramado. É um pouco mais caro do que os outros sopradores 
de folhas disponíveis no mercado, mas acho que vale a pena pelas 
características extras."""


In [ ]:


# Avaliação do Lavador de Cabelo

review_cliente2 = """Eu adorei o lavador de cabelo que comprei! 
A pele de ovo que ele tem é tão suave e absorvente. Usado no cabelo seco 
e oleoso, ele me deixou com cabelos suaves e brilhantes. 
Além disso, a garrafa é extremamente segura e fácil de 
manusear R$ 1200,00 . O que mais me agrada é que ele é ecologicamente 
correto, sem produtos químicos nocivos. Apenas um produto 
de verdade para o meu cabelo. 10/10 recomedado para 
qualquer pessoa procurando um produto de lavagem de cabelo 
de alta qualidade!"""


# Avaliação do Cremedor de Rosto

review_cliente3 = """O cremedor de rosto que adquiri foi perfeitamente 
adequado para os meus objetivos. A facilidade de uso e 
limpeza é impressionante. O que mais me agrada é a capacidade 
de ajustar a temperatura de acordo com as minhas necessidades. 
Ele é um produto essencial para qualquer pessoa que 
quiser manter a pele saudável e radiante. A qualidade é 
superior e a durabilidade é excelente. Recomendo sem 
reservas para todos os interessados em um bom cremedor de rosto."""

In [ ]:
from pydantic import BaseModel, Field

class AvaliacaoReview(BaseModel):
    """Avalia review do cliente"""
    presente: bool = Field(description='Verdadeiro se foi para presente e False se não foi')
    dias_entrega: int = Field(description='Quantos dias para entrega do produto')
    percepcao_valor: list[str] = Field(description='Extraia qualquer frase sobre o valor ou \
    preço do produto. Retorne uma lista.')

llm_estruturada = chat.with_structured_output(AvaliacaoReview)
resposta = llm_estruturada.invoke(review_cliente)
resposta2 = llm_estruturada.invoke(review_cliente2)
resposta3 = llm_estruturada.invoke(review_cliente3)
resposta

AvaliacaoReview(presente=False, dias_entrega=2, percepcao_valor=['mais caro do que os outros', 'valem a pena pelas características extras'])

In [ ]:
print(f"Resposta 1 : {resposta.presente}")
print(f"Resposta 2 : {resposta2.presente}")
print(f"Resposta 3 : {resposta3.presente}")

Resposta 1 : False
Resposta 2 : True
Resposta 3 : True


In [ ]:
print(f"Resposta 1 : {resposta.dias_entrega}")
print(f"Resposta 2 : {resposta2.dias_entrega}")
print(f"Resposta 3 : {resposta3.dias_entrega}")

Resposta 1 : 2
Resposta 2 : 10
Resposta 3 : 5


In [ ]:
print(f"Resposta 1 : {resposta.percepcao_valor}")
print(f"Resposta 2 : {resposta2.percepcao_valor}")
print(f"Resposta 3 : {resposta3.percepcao_valor}")

Resposta 1 : ['mais caro do que os outros', 'valem a pena pelas características extras']
Resposta 2 : ['R$ 1200,00']
Resposta 3 : []
